In [1]:
import time
import logging
import gc
import warnings
import os
import tempfile
import ast
import seaborn as sns
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import shap
import folium
import joblib
from joblib import Parallel, delayed
from tqdm import tqdm
from scipy import stats, optimize
from shapely.geometry import LineString
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler, OneHotEncoder, PolynomialFeatures
from folium.plugins import Draw
from folium import Map, Marker
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, train_test_split

import xgboost as xgb
from xgboost import XGBRegressor, DMatrix, train as xgb_train
from flask import Flask, render_template, request, redirect, url_for
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, Dropout, Concatenate, Embedding, Flatten, Conv1D, MaxPooling1D, Reshape, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback
from tensorflow.keras.regularizers import l1, l2

from IPython.display import display, HTML, IFrame, clear_output
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.gridspec as gridspec
from PIL import Image
from textwrap import wrap

import ipywidgets as widgets

# Import functions from Preprocessing.py and Map_Plotting.py
from Preprocessing import (
    split_parameters, 
    handle_missing_values, drop_na_parameter_columns,
    group_and_summarize, drop_zero_rows, convert_to_absolute_values,
    replace_missing_water_params, replace_zeros_with_P50, assign_basin_tc,
    add_neighbor_eur_cumulative, select_and_drop_columns, load_and_preprocess_data,
    drop_specified_columns
)

from Map_Plotting import create_well_map

In [2]:
# Main script logic
file_path = r'C:\Users\Prakhar.Sarkar\OneDrive - SRP Management Services\Documents\_For_Prakhar\Prakhar_Testnew2.xlsx'
uwi_columns = ['UWI', 'NNAZ_1_UWI', 'NNAZ_2_UWI', 'NNAZ_3_UWI',
               'NNAZ_4_UWI', 'NNAZ_5_UWI', 'NNAZ_6_UWI', 
               'NNSZ_1_UWI', 'NNSZ_2_UWI']

In [3]:
df = load_and_preprocess_data(file_path, uwi_columns)

              UWI             UWI      NNAZ_1_UWI      NNAZ_2_UWI  \
0  42383398110000  42383398110000  42383401300000  42383401290000   
1  42383398120000  42383398120000  42383398130000  42383398010100   
2  42383398130000  42383398130000  42383398140000  42383398120000   
3  42383398140000  42383398140000  42383398130000  42383397870000   
4  42383398150000  42383398150000  42383398160000  42383411170000   

       NNAZ_3_UWI      NNAZ_4_UWI      NNAZ_5_UWI      NNAZ_6_UWI  \
0  42383403500000  42383403510000  42383403510000  42383403510000   
1  42383398000000  42383398140000  42383398140000  42383398140000   
2  42383397870000  42383398010100  42383398010100  42383398010100   
3  42383397880000  42383398120000  42383398120000  42383398120000   
4  42383411150000  42383411160000  42383411160000  42383411160000   

       NNSZ_1_UWI      NNSZ_2_UWI  
0  42383401290000  42383403510000  
1  42383398000000  42383398140000  
2  42383397870000  42383398010100  
3  42383397880000  4238339

In [4]:
param_columns = [
    'Oil_Params_P20', 'Gas_Params_P20', 'Oil_Params_P35', 'Gas_Params_P35', 
    'Oil_Params_P50', 'Gas_Params_P50', 'Oil_Params_P65', 'Gas_Params_P65', 
    'Oil_Params_P80', 'Gas_Params_P80', 'Water_Params_P50'
]

# Apply the functions in sequence
df, new_columns = split_parameters(df, param_columns)
df = handle_missing_values(df)
df = drop_na_parameter_columns(df, new_columns)
grouped_df, uwi10_sum = group_and_summarize(df)

# Display the results
print(grouped_df)
print(uwi10_sum)

              Typecurve  UWI10
0    CoyoteValley_North    100
1    CoyoteValley_South    299
2        FarEast_Howard    131
3                  Hutt    185
4           Mercury_600    112
..                  ...    ...
109            proj96_A     82
110            proj96_B    286
111              proj97    161
112              proj98    282
113              proj99    117

[114 rows x 2 columns]
15500


In [5]:
columns_to_check = ['FluidPerFoot_bblft', 'ProppantPerFoot', 'EUR_30yr_Actual_Gas_P50_MMCF', 
                    'EUR_30yr_Actual_Oil_P50_MBO', 'HEELPOINT_LAT']
df = drop_zero_rows(df, columns_to_check)

df = convert_to_absolute_values(df)
df = replace_missing_water_params(df)
replace_zeros_with_P50(df)
df = add_neighbor_eur_cumulative(df)
df = df[df['BasinTC'] != 'Unknown']
# m = create_well_map(df)
# display(IFrame(m, width=700, height=500))

Data dropped after removing zero 'FluidPerFoot_bblft': 1411
Data dropped after removing zero 'ProppantPerFoot': 187
Data dropped after removing zero 'EUR_30yr_Actual_Gas_P50_MMCF': 4
Data dropped after removing zero 'EUR_30yr_Actual_Oil_P50_MBO': 1
Data dropped after removing zero 'HEELPOINT_LAT': 331


In [6]:
# Call the function and pass the DataFrame
#df = select_and_drop_columns(df)
# df

In [7]:
#Hard Coded for now
columns_to_drop=['UWI10', 'CompletionDate' ,'UWI', 'WellName','NNAZ_1_UWI','NNAZ_2_UWI','NNAZ_3_UWI','NNAZ_4_UWI','NNAZ_5_UWI',
                 'NNAZ_6_UWI','NNSZ_1_UWI',
 'NNSZ_2_UWI','LeaseName', 'WellNumber', 'CurrentOperatorName', 'OriginalOperatorName', 'DrillingContractorName', 'PermitDate', 
                 'SpudDate','FORMATION_CONDENSE', 'Unique_PDP_ID','EUR_30yr_Actual_Oil_P20_MBO',
 'EUR_30yr_Actual_Gas_P20_MMCF', 'EUR_30yr_Actual_Oil_P35_MBO', 'EUR_30yr_Actual_Gas_P35_MMCF', 'EUR_30yr_Actual_Oil_P50_MBO', 
                 'EUR_30yr_Actual_Gas_P50_MMCF',
 'EUR_30yr_Actual_Oil_P65_MBO', 'EUR_30yr_Actual_Gas_P65_MMCF', 'EUR_30yr_Actual_Oil_P80_MBO', 'EUR_30yr_Actual_Gas_P80_MMCF',
                 'EUR_30yr_Actual_Water_P50_MBBL','WELL_TORTUOSITY','DEPTH_TO_TOP_2Q',
 'DEPTH_TO_TOP_3Q', 'DEPTH_TO_TOP_4Q', 'AZIMUTH','DEPTH_ABOVE_ZONE_2Q',  'DEPTH_ABOVE_ZONE_3Q', 'DEPTH_ABOVE_ZONE_4Q','Cumulative oil mbo', 
 'Cumulative gas mmcf', 'Cumulative water mbbl','AVERAGE_INCLINATION','DEPTH_TO_TOP_1Q','HEELPOINT_DEPTH','TOEPOINT_DEPTH', 'NNSZ_1_FORMATION',
 'NNSZ_2_FORMATION'
                 # ,
 # 'NNAZ_3_EUR_30yr_Actual_Oil_P20_MBO', 'NNAZ_3_EUR_30yr_Actual_Oil_P35_MBO', 'NNAZ_3_EUR_30yr_Actual_Oil_P50_MBO',
 # 'NNAZ_3_EUR_30yr_Actual_Oil_P65_MBO', 'NNAZ_3_EUR_30yr_Actual_Oil_P80_MBO', 'NNAZ_3_EUR_30yr_Actual_Gas_P20_MMCF',
 # 'NNAZ_3_EUR_30yr_Actual_Gas_P35_MMCF', 'NNAZ_3_EUR_30yr_Actual_Gas_P50_MMCF', 'NNAZ_3_EUR_30yr_Actual_Gas_P65_MMCF',
 # 'NNAZ_3_EUR_30yr_Actual_Gas_P80_MMCF', 'NNAZ_3_Cumulative oil mbo', 'NNAZ_3_Cumulative gas mmcf', 'NNAZ_3_Cumulative water mbbl',
 # 'NNAZ_4_EUR_30yr_Actual_Oil_P20_MBO', 'NNAZ_4_EUR_30yr_Actual_Oil_P35_MBO', 'NNAZ_4_EUR_30yr_Actual_Oil_P50_MBO',
 # 'NNAZ_4_EUR_30yr_Actual_Oil_P65_MBO', 'NNAZ_4_EUR_30yr_Actual_Oil_P80_MBO', 'NNAZ_4_EUR_30yr_Actual_Gas_P20_MMCF',
 # 'NNAZ_4_EUR_30yr_Actual_Gas_P35_MMCF', 'NNAZ_4_EUR_30yr_Actual_Gas_P50_MMCF', 'NNAZ_4_EUR_30yr_Actual_Gas_P65_MMCF',
 # 'NNAZ_4_EUR_30yr_Actual_Gas_P80_MMCF', 'NNAZ_4_Cumulative oil mbo', 'NNAZ_4_Cumulative gas mmcf',
 # 'NNAZ_4_Cumulative water mbbl', 'NNAZ_5_EUR_30yr_Actual_Oil_P20_MBO', 'NNAZ_5_EUR_30yr_Actual_Oil_P35_MBO',
 # 'NNAZ_5_EUR_30yr_Actual_Oil_P50_MBO', 'NNAZ_5_EUR_30yr_Actual_Oil_P65_MBO', 'NNAZ_5_EUR_30yr_Actual_Oil_P80_MBO',
 # 'NNAZ_5_EUR_30yr_Actual_Gas_P20_MMCF', 'NNAZ_5_EUR_30yr_Actual_Gas_P35_MMCF', 'NNAZ_5_EUR_30yr_Actual_Gas_P50_MMCF',
 # 'NNAZ_5_EUR_30yr_Actual_Gas_P65_MMCF', 'NNAZ_5_EUR_30yr_Actual_Gas_P80_MMCF', 'NNAZ_5_Cumulative oil mbo',
 # 'NNAZ_5_Cumulative gas mmcf', 'NNAZ_5_Cumulative water mbbl', 'NNAZ_6_EUR_30yr_Actual_Oil_P20_MBO',
 # 'NNAZ_6_EUR_30yr_Actual_Oil_P35_MBO', 'NNAZ_6_EUR_30yr_Actual_Oil_P50_MBO', 'NNAZ_6_EUR_30yr_Actual_Oil_P65_MBO',
 # 'NNAZ_6_EUR_30yr_Actual_Oil_P80_MBO', 'NNAZ_6_EUR_30yr_Actual_Gas_P20_MMCF', 'NNAZ_6_EUR_30yr_Actual_Gas_P35_MMCF',
 # 'NNAZ_6_EUR_30yr_Actual_Gas_P50_MMCF', 'NNAZ_6_EUR_30yr_Actual_Gas_P65_MMCF', 'NNAZ_6_EUR_30yr_Actual_Gas_P80_MMCF',
 # 'NNAZ_6_Cumulative oil mbo', 'NNAZ_6_Cumulative gas mmcf', 'NNAZ_6_Cumulative water mbbl','NNSZ_1_FORMATION', 'NNSZ_2_FORMATION',
 #                 'DEPTH_ABOVE_ZONE_1Q',
 # 'NNAZ_1_FORMATION', 'NNAZ_2_FORMATION', 'NNAZ_3_FORMATION', 'NNAZ_4_FORMATION', 'NNAZ_5_FORMATION', 'NNAZ_6_FORMATION',
 # 'NNAZ_1_EUR_30yr_Actual_Oil_P20_MBO', 'NNAZ_1_EUR_30yr_Actual_Oil_P35_MBO',  'NNAZ_1_EUR_30yr_Actual_Oil_P65_MBO',
 # 'NNAZ_1_EUR_30yr_Actual_Oil_P80_MBO', 'NNAZ_1_EUR_30yr_Actual_Gas_P20_MMCF', 'NNAZ_1_EUR_30yr_Actual_Gas_P35_MMCF',
 # 'NNAZ_1_EUR_30yr_Actual_Gas_P65_MMCF', 'NNAZ_1_EUR_30yr_Actual_Gas_P80_MMCF', 'NNAZ_2_EUR_30yr_Actual_Oil_P20_MBO',
 # 'NNAZ_2_EUR_30yr_Actual_Oil_P35_MBO', 'NNAZ_2_EUR_30yr_Actual_Oil_P65_MBO', 'NNAZ_2_EUR_30yr_Actual_Oil_P80_MBO',
 # 'NNAZ_2_EUR_30yr_Actual_Gas_P20_MMCF', 'NNAZ_2_EUR_30yr_Actual_Gas_P35_MMCF', 'NNAZ_2_EUR_30yr_Actual_Gas_P65_MMCF',
 # 'NNAZ_2_EUR_30yr_Actual_Gas_P80_MMCF', 'NNSZ_1_EUR_30yr_Actual_Oil_P20_MBO', 'NNSZ_1_EUR_30yr_Actual_Oil_P35_MBO',
 # 'NNSZ_1_EUR_30yr_Actual_Oil_P65_MBO', 'NNSZ_1_EUR_30yr_Actual_Oil_P80_MBO', 'NNSZ_1_EUR_30yr_Actual_Gas_P20_MMCF',
 # 'NNSZ_1_EUR_30yr_Actual_Gas_P35_MMCF', 'NNSZ_1_EUR_30yr_Actual_Gas_P65_MMCF', 'NNSZ_1_EUR_30yr_Actual_Gas_P80_MMCF',
 # 'NNSZ_2_EUR_30yr_Actual_Oil_P20_MBO', 'NNSZ_2_EUR_30yr_Actual_Oil_P35_MBO', 'NNSZ_2_EUR_30yr_Actual_Oil_P65_MBO',
 # 'NNSZ_2_EUR_30yr_Actual_Oil_P80_MBO', 'NNSZ_2_EUR_30yr_Actual_Gas_P20_MMCF', 'NNSZ_2_EUR_30yr_Actual_Gas_P35_MMCF',
 # 'NNSZ_2_EUR_30yr_Actual_Gas_P65_MMCF', 'NNSZ_2_EUR_30yr_Actual_Gas_P80_MMCF',
 #                 'NNAZ_3_TRUEDIST',
 # 'NNAZ_3_HZDIST',
 # 'NNAZ_3_VTDIST',
 # 'NNAZ_4_TRUEDIST',
 # 'NNAZ_4_HZDIST',
 # 'NNAZ_4_VTDIST',
 # 'NNAZ_5_TRUEDIST',
 # 'NNAZ_5_HZDIST',
 # 'NNAZ_5_VTDIST',
 # 'NNAZ_6_TRUEDIST',
 # 'NNAZ_6_HZDIST',
 # 'NNAZ_6_VTDIST',
# 'Oil_Params_P20_BuildupRate',
#  'Oil_Params_P20_MonthsInProd',
#  'Oil_Params_P20_InitialProd',
#  'Oil_Params_P20_DiCoefficient',
#  'Oil_Params_P20_BCoefficient',
#  'Oil_Params_P20_LimDeclineRate',
#  'Gas_Params_P20_BuildupRate',
#  'Gas_Params_P20_MonthsInProd',
#  'Gas_Params_P20_InitialProd',
#  'Gas_Params_P20_DiCoefficient',
#  'Gas_Params_P20_BCoefficient',
#  'Gas_Params_P20_LimDeclineRate',
#  'Oil_Params_P35_BuildupRate',
#  'Oil_Params_P35_MonthsInProd',
#  'Oil_Params_P35_InitialProd',
#  'Oil_Params_P35_DiCoefficient',
#  'Oil_Params_P35_BCoefficient',
#  'Oil_Params_P35_LimDeclineRate',
#  'Gas_Params_P35_BuildupRate',
#  'Gas_Params_P35_MonthsInProd',
#  'Gas_Params_P35_InitialProd',
#  'Gas_Params_P35_DiCoefficient',
#  'Gas_Params_P35_BCoefficient',
#  'Gas_Params_P35_LimDeclineRate',
#                   'Oil_Params_P65_BuildupRate',
#  'Oil_Params_P65_MonthsInProd',
#  'Oil_Params_P65_InitialProd',
#  'Oil_Params_P65_DiCoefficient',
#  'Oil_Params_P65_BCoefficient',
#  'Oil_Params_P65_LimDeclineRate',
#  'Gas_Params_P65_BuildupRate',
#  'Gas_Params_P65_MonthsInProd',
#  'Gas_Params_P65_InitialProd',
#  'Gas_Params_P65_DiCoefficient',
#  'Gas_Params_P65_BCoefficient',
#  'Gas_Params_P65_LimDeclineRate',
#  'Oil_Params_P80_BuildupRate',
#  'Oil_Params_P80_MonthsInProd',
#  'Oil_Params_P80_InitialProd',
#  'Oil_Params_P80_DiCoefficient',
#  'Oil_Params_P80_BCoefficient',
#  'Oil_Params_P80_LimDeclineRate',
#  'Gas_Params_P80_BuildupRate',
#  'Gas_Params_P80_MonthsInProd',
#  'Gas_Params_P80_InitialProd',
#  'Gas_Params_P80_DiCoefficient',
#  'Gas_Params_P80_BCoefficient',
#  'Gas_Params_P80_LimDeclineRate'
]
# Dropping the columns
df.drop(columns=columns_to_drop, inplace=True)

In [ ]:
df = drop_specified_columns(df)